# T2: Kiểm tra environment và W&B

Notebook chưa được xác nhận chạy Colab end-to-end. Runtime đã quan sát là Python 3.13.18, khác target 3.12; cần chọn runtime tương thích, không bỏ guard.

Trước khi publish: tạo ZIP từ DEVELOPMENT, gồm source/tests/requirements, bỏ .git, venv, cache, W&B logs và secrets. Đường dẫn ZIP Windows dùng backslash được chuẩn hóa, nhưng path traversal, drive path, symlink và đường dẫn trùng vẫn bị chặn.

Sau khi code đã publish: chọn clone và nhập SHA 40 ký tự chứa T2. SOURCE=existing để dùng lại source sau khi restart. Không suy từ branch hoặc tên ZIP rằng source đã đúng phiên bản.

COLAB RUNTIME NOT TESTED. Static checks không chứng minh runtime tương thích. Nếu Torch khác pin, notebook dừng trước cài đặt. Chỉ chủ động cho phép CPU wheel sau khi xem dung lượng; không tự nâng CUDA/driver.


In [ ]:
def extract_source_zip(archive, target):
    """Giai nen source ZIP, doi dau phan cach Windows va chan duong dan nguy hiem."""
    import shutil
    import stat
    from pathlib import Path, PurePosixPath

    target = Path(target).resolve()
    planned = []
    seen = set()
    files = set()
    for entry in archive.infolist():
        name = entry.filename.replace("\\", "/")
        relative = PurePosixPath(name)
        if (not relative.parts or relative.is_absolute()
                or ".." in relative.parts or any(":" in part for part in relative.parts)):
            raise ValueError("Unsafe archive path")
        destination = (target / Path(*relative.parts)).resolve()
        if not destination.is_relative_to(target) or destination == target:
            raise ValueError("Unsafe archive path")
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError("Archive symlinks are not allowed")
        if destination in seen:
            raise ValueError("Duplicate normalized archive path")
        seen.add(destination)
        is_directory = name.endswith("/") or entry.is_dir()
        if not is_directory:
            files.add(destination)
        planned.append((entry, destination, is_directory))
    # Kiem tra het archive truoc khi ghi de tranh giai nen do dang.
    if any(parent in files for _, destination, _ in planned for parent in destination.parents):
        raise ValueError("Archive file conflicts with a parent directory")
    for entry, destination, is_directory in planned:
        if is_directory:
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, destination.open("xb") as output:
                shutil.copyfileobj(source, output)

import os, pathlib, subprocess, sys, zipfile, re
SOURCE = "upload"  # Chon upload, clone hoac existing sau khi restart.
PUBLISHED_SHA = ""  # Can SHA 40 ky tu da publish neu chon clone.
root = pathlib.Path("/content/t2-source")
if SOURCE != "existing":
    if root.exists():
        raise RuntimeError("Source folder already exists. Use existing to reuse it or choose a fresh runtime.")
    root.mkdir()
if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one source zip")
    import io
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        extract_source_zip(archive, root)
    candidates = list(root.rglob("requirements.txt"))
    candidates = [p.parent for p in candidates if (p.parent / "core/tracking.py").exists()]
    if len(candidates) != 1:
        raise RuntimeError("Zip must contain exactly one T2 repo")
    root = candidates[0]
elif SOURCE == "clone":
    if not re.fullmatch(r"[0-9a-fA-F]{40}", PUBLISHED_SHA):
        raise ValueError("Enter the real published T2 commit SHA")
    subprocess.run(["git", "clone", "https://github.com/nvk1701/github-pj.git", str(root)], check=True)
    subprocess.run(["git", "checkout", PUBLISHED_SHA], cwd=root, check=True)
elif SOURCE == "existing":
    candidates = [p.parent for p in root.rglob("requirements.txt") if (p.parent / "core/tracking.py").exists()]
    if len(candidates) != 1:
        raise RuntimeError("Expected exactly one previously extracted T2 source folder")
    root = candidates[0]
else:
    raise ValueError("Choose upload, clone, or existing")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Source:", root)
print("Python:", sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("These pins target Python 3.12; use a compatible runtime before continuing")


In [ ]:
# Doc pins tu requirements; khong tu thay Torch/CUDA.
from importlib.metadata import version, PackageNotFoundError
pins = dict(line.split("==") for line in pathlib.Path("requirements.txt").read_text(encoding="utf-8").splitlines()
            if line.strip() and not line.startswith("#"))
print("Runtime interpreter:", sys.executable)
print("Runtime Python:", sys.version)
ALLOW_CPU_TORCH_INSTALL = False  # Chi bat sau khi da xem tac dong toi runtime va dung luong tai.
try:
    installed = version("torch")
except PackageNotFoundError:
    installed = None
print("Torch installed:", installed, "required:", pins["torch"])
if installed is None or installed.split("+")[0] != pins["torch"]:
    if not ALLOW_CPU_TORCH_INSTALL:
        raise RuntimeError("Torch pin mismatch: select a matching runtime, or explicitly allow the pinned CPU wheel. No dependency has been silently skipped. GPU validation requires a separately compatible build/runtime.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                    "torch==" + pins["torch"], "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Neu package da import bi thay, restart va chon SOURCE="existing".


In [ ]:
subprocess.run([sys.executable, "-m", "scripts.check_environment", "--gpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
# Test SDK offline chay trong process rieng, khong login hay upload.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Smoke online bằng tài khoản riêng

Entity `gr_ml_29_37_37`, project `ml-distribution-shift`.
[Dashboard](https://forge.coreweave.com/wandb/gr_ml_29_37_37/ml-distribution-shift).
PUBLIC và [run online trên VS Code](https://forge.coreweave.com/wandb/gr_ml_29_37_37/ml-distribution-shift/runs/kc5iilt2) do người dùng xác nhận. Chưa có đủ run của cả ba tài khoản; chưa nghiệm thu Colab end-to-end.

Lưu WANDB_API_KEY trong Colab Secrets và cấp quyền notebook. Mỗi thành viên dùng key riêng, không dán key vào cell. Điền MEMBER; chạy cell online sau khi checks offline đạt. Lưu URL và kiểm uploader thực tế; member label không chứng minh danh tính.


In [ ]:
from google.colab import userdata
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
import wandb
TEAM_ENTITY = "gr_ml_29_37_37"  # Entity cua team da xac nhan.
PROJECT = "ml-distribution-shift"
MEMBER = ""  # Dien ten cua chinh ban.
if not TEAM_ENTITY.strip() or not MEMBER.strip():
    raise ValueError("Fill in confirmed TEAM_ENTITY and your MEMBER name")
# SDK 0.31.0 nhan key moi va mac dinh dung Forge API.
# Dung API host, khong dung URL dashboard; restart kernel sau khi nang SDK.
wandb.login(key=userdata.get("WANDB_API_KEY"), host="https://api.forge.coreweave.com",
            relogin=True, verify=True)
os.environ["WANDB_ENTITY"] = TEAM_ENTITY
os.environ["WANDB_PROJECT"] = PROJECT
from core.tracking import init_run
import random
rng = random.Random(42)
run = init_run(dict(project=PROJECT, entity=TEAM_ENTITY, mode="online", seed=42,
                    method="smoke-test", member=MEMBER,
                    name=f"smoke-{MEMBER}-seed42", tags=["smoke-test", "colab"]))
try:
    assert wandb.run is run
    for step in range(3):
        metric = {"smoke/value": rng.random(), "smoke/step": step}
        if step == 0:
            wandb.log(metric, step=step, commit=True)
        else:
            run.log(metric, step=step, commit=True)
    run.summary["smoke/completed"] = True
except BaseException:
    try:
        run.finish(exit_code=1)
    except Exception:
        pass  # Giu loi ban dau neu cleanup cung loi.
    raise
else:
    run.finish()
print("Run URL:", run.url)
